<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Deep Learning for MNIST Classification — Algorithm</b></h1>
</div>


## Purpose and reading order

This notebook is the algorithm specification for the corresponding `main.ipynb` in this lab. It has two linked layers:

| Layer | Content | Question answered |
| --- | --- | --- |
| **Part I — Complete pseudocode** | One self-contained executable-style block with explicit branches and gates | What is the whole pipeline? |
| **Part II — Auditable specification** | Six-section contract for every step: Purpose, Inputs, Procedure, Produces, Acceptance, Failure mode | Why is each step valid? |

Names, shapes, thresholds, and failure decisions are kept identical between both layers. The acceptance contract of step *k* is the precondition of step *k+1*.

## Part I — Complete Pseudocode

The single block below is the complete pipeline. Every named artifact is assigned before use; every failure is explicit. It is a direct, implementation-neutral transcription of the executable notebook.

```text
ALGORITHM MNIST_MLP_Training

# Algorithm 1 — Setup — Environment and Configuration
SEED ← 42
CALL ARRAY.random.seed(SEED)
CALL TENSOR.manual_seed(SEED)
device ← TENSOR.device((IF TENSOR.cuda.is_available() THEN "cuda" ELSE "cpu"))
BATCH_SIZE ← 512
HIDDEN_LAYER_WIDTHS ← [128, 256, 512]
EPOCHS ← 10
LEARNING_RATE ← 0.001
WEIGHT_DECAY ← 0.001
N_CLASSES ← 10
IMAGE_SIZE ← 28
INPUT_DIM ← (IMAGE_SIZE * IMAGE_SIZE)
OUTPUT_DIR ← PATH("../outputs/figures")
CALL OUTPUT_DIR.mkdir(parents = TRUE, exist_ok = TRUE)
CALL EMIT(FORMAT("PyTorch version: {0}" , TENSOR.__version__))
CALL EMIT(FORMAT("Device: {0}" , device))

# Algorithm 2 — 1. Validate MNIST Data and Output Paths
DATA_DIR ← PATH("../data")
discovered_mnist_files ← MAP{path.name ↦ path FOR path IN DATA_DIR.glob("*-ubyte") WHERE path.is_file()}
REQUIRED_MNIST_FILES ← {"train-images-idx3-ubyte", "train-labels-idx1-ubyte", "t10k-images-idx3-ubyte",
    "t10k-labels-idx1-ubyte"}
missing_mnist_files ← SORT((REQUIRED_MNIST_FILES − SET(discovered_mnist_files)))
IF missing_mnist_files THEN
    RAISE FileNotFoundError(("Missing MNIST IDX files: " + ", ".join(missing_mnist_files)))
END IF
CALL OUTPUT_DIR.mkdir(parents = TRUE, exist_ok = TRUE)
CALL EMIT(FORMAT("MNIST data directory : {0}" , DATA_DIR))
CALL EMIT(FORMAT("IDX files discovered : {0}" , LENGTH(discovered_mnist_files)))
CALL EMIT("Required containers  : 4/4")
CALL EMIT(FORMAT("Figures directory    : {0}" , OUTPUT_DIR))

# Algorithm 3 — 2. Build the MNIST Dataset and Mini-Batch Pipeline
CLASS MNISTDataset EXTENDS DATASET_INTERFACE
    image_size ← IMAGE_SIZE
    PROCEDURE __init__(self, partition, mnist_directory)
        IF (partition NOT IN ("training", "testing")) THEN
            RAISE ValueError(FORMAT("{0!r} is not a valid partition." , partition))
        END IF
        mnist ← READ_MNIST_IDX(STRING(mnist_directory))
        parser ← GET_ATTRIBUTE(mnist, FORMAT("load_{0}" , partition))
        (self.images, self.labels) ← parser()
        self.nimages ← LENGTH(self.images)
    END PROCEDURE
    PROCEDURE __len__(self)
        RETURN self.nimages
    END PROCEDURE
    PROCEDURE __getitem__(self, index)
        image ← (ARRAY.asarray(self.images[index], dtype = ARRAY.float32) / 255.0)
        label ← INTEGER(self.labels[index])
        RETURN {"image" ↦ image, "label" ↦ label}
    END PROCEDURE
    STATIC PROCEDURE collate_fn(data_batch)
        images ← CAST(ARRAY.stack(LIST[item["image"] FOR item IN data_batch], axis = 0), ARRAY.float32)
        labels ← ARRAY.asarray(LIST[item["label"] FOR item IN data_batch], dtype = ARRAY.int64)
        RETURN {"image" ↦ images, "label" ↦ labels}
    END PROCEDURE
END CLASS
PROCEDURE build_mnist_dataset_and_loader(batch_size, partition, dataset_directory)
    dataset ← MNISTDataset(partition = partition, mnist_directory = dataset_directory)
    loader ← BATCH_LOADER(dataset, batch_size = batch_size, shuffle = (partition = "training"),
        num_workers = 0, collate_fn = dataset.collate_fn)
    RETURN (dataset, loader)
END PROCEDURE
CALL EMIT("MNIST dataset and DataLoader pipeline defined.")

# Algorithm 4 — 3. Load and Validate Training and Testing Data
(train_dataset, training_data_loader) ← build_mnist_dataset_and_loader(batch_size = BATCH_SIZE,
    partition = "training", dataset_directory = DATA_DIR)
(test_dataset, test_data_loader) ← build_mnist_dataset_and_loader(batch_size = BATCH_SIZE, partition = "testing",
    dataset_directory = DATA_DIR)
IF (LENGTH(train_dataset) ≠ 60000) THEN
    RAISE ValueError(FORMAT("Expected 60,000 training images; found {0:,}." , LENGTH(train_dataset)))
END IF
IF (LENGTH(test_dataset) ≠ 10000) THEN
    RAISE ValueError(FORMAT("Expected 10,000 test images; found {0:,}." , LENGTH(test_dataset)))
END IF
sample ← train_dataset[0]
IF (sample["image"].shape ≠ (INPUT_DIM,)) THEN
    RAISE ValueError(FORMAT("Expected flattened image shape {0}; found {1}." , (INPUT_DIM,),
        sample["image"].shape))
END IF
IF (NOT (0 ≤ sample["label"] < N_CLASSES)) THEN
    RAISE ValueError("MNIST label is outside the expected range 0–9.")
END IF
IF ((MIN(sample["image"]) < 0.0) OR (MAX(sample["image"]) > 1.0)) THEN
    RAISE ValueError("MNIST pixels are not normalized to [0, 1].")
END IF
CALL EMIT(FORMAT("Training images: {0:,}" , LENGTH(train_dataset)))
CALL EMIT(FORMAT("Testing images: {0:,}" , LENGTH(test_dataset)))
CALL EMIT(FORMAT("Flattened image shape: {0}" , sample["image"].shape))
CALL EMIT(FORMAT("Pixel range: [{0:.3f}, {1:.3f}]" , MIN(sample["image"]), MAX(sample["image"])))

# Algorithm 5 — 4. Visualize Representative MNIST Samples
sample_indices ← ARRAY.arange(12)
(fig, axes) ← PLOT.subplots(3, 4, figsize = (8, 6))
axes ← RESHAPE(axes, (−1))
FOR (ax, index) IN ZIP(axes, sample_indices) DO
    item ← train_dataset[index]
    CALL ax.imshow(RESHAPE(item["image"], IMAGE_SIZE, IMAGE_SIZE), cmap = "gray")
    CALL ax.set_title(FORMAT("Label: {0}" , item["label"]))
    CALL ax.axis("off")
END FOR
CALL PLOT.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "mnist_sample_batch.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()

# Algorithm 6 — 5. Define the One-Hidden-Layer MLP Classifier
CLASS MNISTClassifier EXTENDS LAYERS.Module
    PROCEDURE __init__(self, hidden_layer_width, nclasses = N_CLASSES, imsize = IMAGE_SIZE,
        use_batch_norm = TRUE)
        CALL super().__init__()
        layers ← [LAYERS.Linear((imsize * imsize), hidden_layer_width)]
        IF use_batch_norm THEN
            CALL APPEND(layers, LAYERS.BatchNorm1d(hidden_layer_width))
        END IF
        CALL EXTEND(layers, [LAYERS.ReLU(), LAYERS.Linear(hidden_layer_width, nclasses)])
        self.net ← LAYERS.Sequential(EXPAND layers)
        self.loss_fn ← LAYERS.functional.cross_entropy
    END PROCEDURE
    PROCEDURE forward(self, data_dict)
        logits ← self.net(data_dict["image"])
        IF self.training THEN
            loss ← self.loss_fn(logits, data_dict["label"])
            RETURN {"loss" ↦ loss, "logits" ↦ logits}
        END IF
        (predicted_class, confidence) ← self.decode_prediction(logits)
        RETURN {"cls" ↦ predicted_class, "prob" ↦ confidence, "logits" ↦ logits}
    END PROCEDURE
    STATIC PROCEDURE decode_prediction(logits)
        probabilities ← TENSOR.softmax(logits, dim = 1)
        (confidence, predicted_class) ← MAX(TENSOR, probabilities, dim = 1)
        RETURN (predicted_class, confidence)
    END PROCEDURE
END CLASS
CALL EMIT("MNISTClassifier defined.")

# Algorithm 7 — 6. Verify the Model Architecture and Forward Pass
sample_classifier ← TO_DEVICE(MNISTClassifier(hidden_layer_width = 256), device)
CALL EMIT(sample_classifier)
sample_batch ← next(iter(training_data_loader))
sample_batch ← {"image" ↦ TO_DEVICE(TENSOR.from_numpy(sample_batch["image"]), device),
    "label" ↦ TO_DEVICE(TENSOR.from_numpy(sample_batch["label"]), device)}
CALL SET_MODE(sample_classifier, TRAINING)
sample_logits ← sample_classifier(sample_batch)
IF (NOT TENSOR.isfinite(sample_logits["loss"])) THEN
    RAISE ValueError("Example forward pass produced a non-finite loss.")
END IF
expected_shape ← (sample_batch["image"].shape[0], N_CLASSES)
IF (TUPLE(sample_logits["logits"].shape) ≠ expected_shape) THEN
    RAISE ValueError(FORMAT("Unexpected logit shape: {0}" , TUPLE(sample_logits["logits"].shape)))
END IF
CALL EMIT(FORMAT("Example training loss: {0:.4f}" , SCALAR_VALUE(sample_logits["loss"])))
CALL EMIT(FORMAT("Logit shape: {0}" , TUPLE(sample_logits["logits"].shape)))

# Algorithm 8 — 7. Define Training and Evaluation Utilities
PROCEDURE prepare_training_batch(batch, compute_device)
    RETURN {"image" ↦ TO_DEVICE(TENSOR.from_numpy(batch["image"]), compute_device),
        "label" ↦ TO_DEVICE(TENSOR.from_numpy(batch["label"]), compute_device)}
END PROCEDURE
PROCEDURE train_classifier(model, training_data_loader, optimizer, epochs, compute_device)
    epoch_loss_history ← []
    FOR epoch IN RANGE(epochs) DO
        CALL SET_MODE(model, TRAINING)
        accumulated_loss ← 0.0
        processed_sample_count ← 0
        progress ← PROGRESS(training_data_loader, desc = FORMAT("Epoch {0}/{1}" , (epoch + 1), epochs),
            leave = FALSE)
        FOR batch IN progress DO
            batch ← prepare_training_batch(batch, compute_device)
            CALL optimizer.zero_grad()
            output ← model(batch)
            loss ← output["loss"]
            CALL loss.backward()
            CALL optimizer.step()
            batch_sample_count ← batch["image"].shape[0]
            accumulated_loss ← (accumulated_loss + (SCALAR_VALUE(loss) * batch_sample_count))
            processed_sample_count ← (processed_sample_count + batch_sample_count)
        END FOR
        epoch_mean_loss ← (accumulated_loss / processed_sample_count)
        CALL APPEND(epoch_loss_history, epoch_mean_loss)
    END FOR
    RETURN epoch_loss_history
END PROCEDURE
NO_GRADIENT PROCEDURE evaluate_classifier(model, data_loader, compute_device)
    CALL SET_MODE(model, EVALUATION)
    all_classes ← []
    all_probabilities ← []
    all_labels ← []
    all_logits ← []
    FOR batch IN data_loader DO
        batch ← prepare_training_batch(batch, compute_device)
        output ← model(batch)
        CALL APPEND(all_classes, TO_CPU(output["cls"]))
        CALL APPEND(all_probabilities, TO_CPU(output["prob"]))
        CALL APPEND(all_labels, TO_CPU(batch["label"]))
        CALL APPEND(all_logits, TO_CPU(output["logits"]))
    END FOR
    classes ← TENSOR.cat(all_classes)
    probabilities ← TENSOR.cat(all_probabilities)
    labels ← TENSOR.cat(all_labels)
    logits ← TENSOR.cat(all_logits)
    accuracy ← SCALAR_VALUE(MEAN((classes = labels).float()))
    RETURN {"cls" ↦ classes, "prob" ↦ probabilities, "label" ↦ labels, "logits" ↦ logits,
        "accuracy" ↦ accuracy}
END PROCEDURE
CALL EMIT("Training and evaluation utilities defined.")

# Algorithm 9 — 8. Train the 128-, 256-, and 512-Neuron Models
models ← {}
training_histories ← {}
model_evaluation_results ← {}
FOR hidden_layer_width IN HIDDEN_LAYER_WIDTHS DO
    CALL EMIT(FORMAT("\nTraining model: {0} → {1} → {2}" , INPUT_DIM, hidden_layer_width, N_CLASSES))
    CALL TENSOR.manual_seed(SEED)
    model ← TO_DEVICE(MNISTClassifier(hidden_layer_width = hidden_layer_width), device)
    optimizer ← TENSOR.optim.Adam(model.parameters(), lr = LEARNING_RATE, weight_decay = WEIGHT_DECAY)
    losses ← train_classifier(model = model, training_data_loader = training_data_loader,
        optimizer = optimizer, epochs = EPOCHS, compute_device = device)
    evaluation_result ← evaluate_classifier(model = model, data_loader = test_data_loader,
        compute_device = device)
    models[hidden_layer_width] ← model
    training_histories[hidden_layer_width] ← losses
    model_evaluation_results[hidden_layer_width] ← evaluation_result
    CALL EMIT(FORMAT("Final training loss: {0:.4f}" , losses[(−1)]))
    CALL EMIT(FORMAT("Test accuracy: {0:.4f}" , evaluation_result["accuracy"]))
END FOR

# Algorithm 10 — 9. Compare Training Loss and Test Accuracy
(fig, ax) ← PLOT.subplots(figsize = (9, 5))
FOR hidden_layer_width IN HIDDEN_LAYER_WIDTHS DO
    losses ← training_histories[hidden_layer_width]
    CALL ax.plot(RANGE(1, (EPOCHS + 1)), losses, marker = "o", label = FORMAT("{0} hidden neurons" ,
        hidden_layer_width))
END FOR
CALL ax.set_title("Training Loss by Hidden-Layer Size")
CALL ax.set_xlabel("Epoch")
CALL ax.set_ylabel("Mean Cross-Entropy Loss")
CALL ax.set_xticks(RANGE(1, (EPOCHS + 1)))
CALL ax.grid(alpha = 0.25)
CALL ax.legend()
CALL PLOT.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "training_loss_comparison.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
REFERENCE_REPORTED_LOSSES ← {128 ↦ 0.0753, 256 ↦ 0.0344, 512 ↦ 0.0394}
CALL EMIT(FORMAT("{0:>8} {1:>12} {2:>15} {3:>15}" , "Hidden", "Final loss", "Test accuracy",
    "Reference loss"))
CALL EMIT(("-" * 55))
FOR hidden_layer_width IN HIDDEN_LAYER_WIDTHS DO
    final_loss ← training_histories[hidden_layer_width][(−1)]
    test_accuracy ← model_evaluation_results[hidden_layer_width]["accuracy"]
    CALL EMIT(FORMAT("{0:>8d} {1:>12.4f} {2:>15.4f} {3:>15.4f}" , hidden_layer_width, final_loss,
        test_accuracy, REFERENCE_REPORTED_LOSSES[hidden_layer_width]))
END FOR

# Algorithm 11 — 10. Visualize Predictions and Confidence Scores
PROCEDURE plot_classifier_predictions(model, dataset, hidden_layer_width, output_path, n_images = 15)
    CALL SET_MODE(model, EVALUATION)
    images ← CAST(ARRAY.stack(LIST[dataset[index]["image"] FOR index IN RANGE(n_images)]), ARRAY.float32)
    labels ← ARRAY.asarray(LIST[dataset[index]["label"] FOR index IN RANGE(n_images)], dtype = ARRAY.int64)
    batch ← {"image" ↦ TO_DEVICE(TENSOR.from_numpy(images), device),
        "label" ↦ TO_DEVICE(TENSOR.from_numpy(labels), device)}
    WITH GRADIENT_RECORDING_DISABLED DO
        output ← model(batch)
    END WITH
    probabilities ← TO_ARRAY(TO_CPU(TENSOR.softmax(output["logits"], dim = 1)))
    fig ← PLOT.figure(figsize = (12, 10))
    FOR index IN RANGE(n_images) DO
        image_axis ← PLOT.subplot(5, 6, ((2 * index) + 1))
        CALL image_axis.imshow(RESHAPE(images[index], IMAGE_SIZE, IMAGE_SIZE), cmap = "gray")
        predicted_label ← INTEGER(ARRAY.argmax(probabilities[index]))
        confidence ← REAL(MAX(ARRAY, probabilities[index]))
        true_label ← INTEGER(labels[index])
        CALL image_axis.set_xlabel(FORMAT("Pred {0} ({1:.0%})\nTrue {2}" , predicted_label, confidence,
            true_label))
        CALL image_axis.set_xticks([])
        CALL image_axis.set_yticks([])
        value_axis ← PLOT.subplot(5, 6, ((2 * index) + 2))
        CALL value_axis.bar(RANGE(N_CLASSES), probabilities[index])
        CALL value_axis.set_ylim(0, 1)
        CALL value_axis.set_xticks(RANGE(N_CLASSES))
        CALL value_axis.set_yticks([])
    END FOR
    CALL fig.suptitle(FORMAT("MNIST Predictions — {0} Hidden Neurons" , hidden_layer_width), y = 1.01)
    CALL PLOT.tight_layout()
    CALL fig.savefig(output_path, dpi = 300, bbox_inches = "tight")
    CALL PLOT.show()
END PROCEDURE
FOR hidden_layer_width IN HIDDEN_LAYER_WIDTHS DO
    CALL plot_classifier_predictions(model = models[hidden_layer_width], dataset = test_dataset,
        hidden_layer_width = hidden_layer_width, output_path = (OUTPUT_DIR / FORMAT("mnist_predictions_{0}.png" ,
        hidden_layer_width)))
END FOR

# Algorithm 12 — 11. Compute Confidence-Threshold Precision, Recall, and Accepted Accuracy
PROCEDURE compute_confidence_threshold_curves(evaluation_result, number_of_thresholds = 100)
    predicted_classes ← TO_ARRAY(evaluation_result["cls"])
    confidence ← TO_ARRAY(evaluation_result["prob"])
    labels ← TO_ARRAY(evaluation_result["label"])
    correct ← (predicted_classes = labels)
    thresholds ← ARRAY.linspace(0.0, 0.99, number_of_thresholds)
    precision_values ← []
    recall_values ← []
    accepted_accuracy_values ← []
    total_correct ← SUM(ARRAY, correct)
    FOR threshold IN thresholds DO
        accepted ← (confidence > threshold)
        true_positive ← SUM(ARRAY, (accepted AND_EACH correct))
        false_positive ← SUM(ARRAY, (accepted AND_EACH (COMPLEMENT correct)))
        false_negative ← SUM(ARRAY, ((COMPLEMENT accepted) AND_EACH correct))
        precision_denominator ← (true_positive + false_positive)
        recall_denominator ← (true_positive + false_negative)
        IF (precision_denominator > 0) THEN
            precision ← (true_positive / precision_denominator)
        ELSE
            precision ← ARRAY.nan
        END IF
        IF (recall_denominator > 0) THEN
            recall ← (true_positive / recall_denominator)
        ELSE
            recall ← 0.0
        END IF
        accepted_accuracy ← (true_positive / LENGTH(labels))
        CALL APPEND(precision_values, precision)
        CALL APPEND(recall_values, recall)
        CALL APPEND(accepted_accuracy_values, accepted_accuracy)
    END FOR
    precision_values ← ARRAY.asarray(precision_values)
    recall_values ← ARRAY.asarray(recall_values)
    accepted_accuracy_values ← ARRAY.asarray(accepted_accuracy_values)
    order ← ARRAY.argsort(recall_values)
    RETURN {"thresholds" ↦ thresholds[order], "precision" ↦ precision_values[order],
        "recall" ↦ recall_values[order], "accepted_accuracy" ↦ accepted_accuracy_values[order],
        "total_correct" ↦ INTEGER(total_correct)}
END PROCEDURE
CALL EMIT("Confidence-threshold evaluation utility defined.")

# Algorithm 13 — 12. Analyze the Best Model and Save Evaluation Curves
best_hidden_size ← MIN(HIDDEN_LAYER_WIDTHS, key = FUNCTION(size) ↦ training_histories[size][(−1)])
best_evaluation ← model_evaluation_results[best_hidden_size]
confidence_curves ← compute_confidence_threshold_curves(best_evaluation, number_of_thresholds = 100)
(fig, axes) ← PLOT.subplots(1, 2, figsize = (11, 4.5))
CALL axes[0].plot(confidence_curves["recall"], confidence_curves["precision"])
CALL axes[0].set_title("Precision–Recall Curve")
CALL axes[0].set_xlabel("Recall")
CALL axes[0].set_ylabel("Precision")
CALL axes[0].grid(alpha = 0.25)
CALL axes[1].plot(confidence_curves["recall"], confidence_curves["accepted_accuracy"])
CALL axes[1].set_title("Accepted Accuracy–Recall Curve")
CALL axes[1].set_xlabel("Recall")
CALL axes[1].set_ylabel("Accepted correct / all test samples")
CALL axes[1].grid(alpha = 0.25)
CALL fig.suptitle(FORMAT("Confidence-Threshold Evaluation — {0} Hidden Neurons" , best_hidden_size))
CALL PLOT.tight_layout()
CALL fig.savefig((OUTPUT_DIR / "pr_accuracy_curve.png"), dpi = 300, bbox_inches = "tight")
CALL PLOT.show()
CALL EMIT(FORMAT("Selected hidden size: {0}" , best_hidden_size))
CALL EMIT(FORMAT("Final training loss: {0:.4f}" , training_histories[best_hidden_size][(−1)]))
CALL EMIT(FORMAT("Test accuracy: {0:.4f}" , best_evaluation["accuracy"]))

# Algorithm 14 — 13. Run Numerical and Output-file Validation Checks
IF (LENGTH(train_dataset) ≠ 60000) THEN
    RAISE ValueError("Training dataset size changed unexpectedly.")
END IF
IF (LENGTH(test_dataset) ≠ 10000) THEN
    RAISE ValueError("Test dataset size changed unexpectedly.")
END IF
IF (SET(models) ≠ SET(HIDDEN_LAYER_WIDTHS)) THEN
    RAISE ValueError("Not all required MLP configurations were trained.")
END IF
FOR hidden_layer_width IN HIDDEN_LAYER_WIDTHS DO
    losses ← ARRAY.asarray(training_histories[hidden_layer_width], dtype = REAL)
    IF (losses.shape ≠ (EPOCHS,)) THEN
        RAISE ValueError(FORMAT("Unexpected loss-history length for hidden size {0}." , hidden_layer_width))
    END IF
    IF (NOT ARRAY.all(ARRAY.isfinite(losses))) THEN
        RAISE ValueError(FORMAT("Non-finite training loss for hidden size {0}." , hidden_layer_width))
    END IF
    evaluation_result ← model_evaluation_results[hidden_layer_width]
    IF (NOT (0.0 ≤ evaluation_result["accuracy"] ≤ 1.0)) THEN
        RAISE ValueError(FORMAT("Invalid test accuracy for hidden size {0}." , hidden_layer_width))
    END IF
    IF (evaluation_result["cls"].shape[0] ≠ LENGTH(test_dataset)) THEN
        RAISE ValueError(FORMAT("Incomplete test predictions for hidden size {0}." , hidden_layer_width))
    END IF
    probabilities ← TO_ARRAY(evaluation_result["prob"])
    IF (NOT ARRAY.all(ARRAY.isfinite(probabilities))) THEN
        RAISE ValueError(FORMAT("Non-finite confidence values for hidden size {0}." , hidden_layer_width))
    END IF
    IF (ARRAY.any((probabilities < 0.0)) OR ARRAY.any((probabilities > 1.0))) THEN
        RAISE ValueError(FORMAT("Confidence values outside [0, 1] for hidden size {0}." ,
            hidden_layer_width))
    END IF
END FOR
REQUIRED_OUTPUTS ← ["mnist_sample_batch.png", "training_loss_comparison.png", "mnist_predictions_128.png", "mnist_predictions_256.png",
    "mnist_predictions_512.png", "pr_accuracy_curve.png"]
missing_outputs ← LIST[name FOR name IN REQUIRED_OUTPUTS WHERE (NOT (OUTPUT_DIR / name).exists())]
IF missing_outputs THEN
    RAISE FileNotFoundError(("Missing outputs: " + ", ".join(missing_outputs)))
END IF
CALL EMIT("All Deep Learning validation checks passed.")

END ALGORITHM
```

## Part II — Step-by-step elaboration

Each step below is a six-section contract corresponding to the numbered block in Part I:

- **Purpose** — why the step exists.
- **Inputs** — artifacts and shapes required before execution.
- **Procedure** — ordered computation.
- **Produces** — outputs and invariants.
- **Acceptance contract** — numerical or structural gates before the next step.
- **Failure mode** — explicit abort, skip, or diagnostic action; never a silent substitution.

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Select the project `.venv` kernel in VS Code.
2. Use this notebook only as the implementation recipe for `main.ipynb`.
3. Keep training, testing, and confidence analysis separate so evaluation remains out-of-sample.

**Produces**

Selected environment and a top-to-bottom execution plan.

**Acceptance contract**

Review the local assumptions before using cached notebook outputs.

**Failure mode**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create the Reproducible Setup

**Purpose**

Make architecture comparisons repeatable.

**Inputs**

PyTorch environment and declared hyperparameters.

**Procedure**

1. Import `Path`, NumPy, Matplotlib, PyTorch, `python-mnist`, `Dataset`, `DataLoader`, and `tqdm`.
2. Set one fixed random seed in NumPy and PyTorch.
3. Select CUDA when available, otherwise CPU.
4. Define batch size, hidden widths `[128,256,512]`, epochs, learning rate, image size, class count, and flattened input dimension.

**Produces**

Seed 42; device; batch 512; 10 epochs; widths 128/256/512.

**Acceptance contract**

Use learning rate 1e-3 and weight decay 1e-3 consistently; a seed alone does not guarantee identical GPU runs.

**Failure mode**

Resolve dependency/device failures before training.

## 2. Validate MNIST Files and Output Paths

**Purpose**

Use the intended local MNIST split.

**Inputs**

DATA_DIR and four exact IDX filenames.

**Procedure**

1. Point `DATA_DIR` to the local MNIST folder and create `OUTPUT_DIR`.
2. Discover the four required IDX files by exact filename.
3. Stop if any file is missing.
4. Keep all generated figures inside the lab output directory.

**Produces**

Verified file inventory; figure directory.

**Acceptance contract**

Both image and label files exist for training and testing.

**Failure mode**

Raise the existing missing-file error rather than replacing a split silently.

## 3. Implement the MNIST Dataset

**Purpose**

Provide one consistent sample representation.

**Inputs**

Training or testing IDX partition.

**Procedure**

1. Create a `Dataset` subclass receiving `training` or `testing` plus the MNIST directory.
2. Load the requested IDX partition.
3. Store images and labels and return partition length in `__len__`.
4. In `__getitem__`, convert one image to a normalized float32 vector and return it with its integer label.

**Produces**

MNISTDataset; image vectors (784,) float32 and integer labels.

**Acceptance contract**

Pixels divided by 255; labels from 0 to 9; sample counts match labels.

**Failure mode**

Reject unsupported partition names; inspect corrupt or mispaired IDX data.

## 4. Build DataLoaders

**Purpose**

Feed mini-batches without mixing evaluation and training.

**Inputs**

MNISTDataset and batch size.

**Procedure**

1. Create a helper that wraps a dataset in a DataLoader.
2. Shuffle training but keep testing deterministic.
3. Create training/testing datasets and loaders with the same batch size.
4. Verify exactly 60,000 training samples and 10,000 test samples.

**Produces**

Training/testing DataLoaders.

**Acceptance contract**

60,000 training and 10,000 testing samples; only training is shuffled.

**Failure mode**

Stop on split-size or batching errors.

## 5. Inspect Representative Samples

**Purpose**

Check that input preprocessing preserves digit meaning.

**Inputs**

Deterministic training sample indices.

**Procedure**

1. Select a deterministic subset of training indices.
2. Reshape flattened vectors to 28×28 only for visualization.
3. Plot images with labels and save the diagnostic figure.
4. Use this to catch wrong normalization, shapes, or labels before training.

**Produces**

28 by 28 sample grid with labels.

**Acceptance contract**

Displayed image and label refer to the same sample; flattening is reversed only for display.

**Failure mode**

Correct ordering or normalization before fitting models.

## 6. Define the MLP Classifier

**Purpose**

Define a controlled width-only comparison.

**Inputs**

784 features, hidden width, 10 classes.

**Procedure**

1. Create a module parameterized by hidden-layer width.
2. Build `784 → hidden_width → 10` linear layers.
3. Insert BatchNorm and ReLU between linear layers.
4. Use cross-entropy directly on logits during training.
5. During inference apply softmax only to decode class and confidence.

**Produces**

MNISTClassifier with Linear, BatchNorm, ReLU and Linear.

**Acceptance contract**

Output logits have 10 columns; cross-entropy receives logits, not softmax probabilities.

**Failure mode**

Correct layer dimensions or loss inputs before training.

## 7. Verify One Forward Pass

**Purpose**

Catch model/batch incompatibility cheaply.

**Inputs**

One training batch and a model on the selected device.

**Procedure**

1. Instantiate one representative model and move it to the selected device.
2. Read one training batch and move tensors to the same device.
3. Run one forward pass in training mode.
4. Verify finite scalar loss and logits shaped `(batch,10)` before launching full training.

**Produces**

Scalar loss and logits shaped (batch, 10).

**Acceptance contract**

Finite loss/logits; labels are int64; batch and model share the device.

**Failure mode**

Stop before the epoch loop if shapes, dtype or device disagree.

## 8. Implement Batch Preparation

**Purpose**

Share batch conversion across training and inference.

**Inputs**

NumPy image and label batches.

**Procedure**

1. Create one helper converting image and label batches to tensors on the requested device.
2. Preserve image float type and integer class labels.
3. Use this helper in both training and evaluation.

**Produces**

Device tensors under image and label keys.

**Acceptance contract**

Images remain float32 and labels int64; batch lengths agree.

**Failure mode**

Correct data preparation instead of casting predictions to repair errors.

## 9. Implement the Training Loop

**Purpose**

Update weights and measure sample-weighted loss.

**Inputs**

Training loader, model, Adam optimizer, epoch count.

**Procedure**

1. For each epoch set `model.train()`.
2. For each batch: prepare tensors, zero gradients, compute loss, backpropagate, and step the optimizer.
3. Accumulate loss weighted by batch size.
4. Store one mean training loss per epoch and return the full history.

**Produces**

epoch_loss_history with one value per epoch.

**Acceptance contract**

Zero gradients each batch; backpropagate only training loss; divide weighted loss by processed sample count.

**Failure mode**

Stop for nonfinite loss; investigate optimization settings rather than fabricate a decreasing history.

## 10. Implement the Evaluation Loop

**Purpose**

Measure predictions without changing model statistics.

**Inputs**

Trained model and testing loader.

**Procedure**

1. Use `torch.no_grad()` and call `model.eval()`.
2. For each test batch collect predicted classes, confidences, labels, and logits on CPU.
3. Concatenate batch outputs.
4. Compute test accuracy and return all evaluation arrays plus accuracy.

**Produces**

cls, prob, label, logits, accuracy.

**Acceptance contract**

eval mode and no_grad; 10,000 aligned predictions; confidence in [0,1].

**Failure mode**

Restore eval mode or alignment before interpreting accuracy.

## 11. Train the Three Width Variants

**Purpose**

Run the same experiment for all three widths.

**Inputs**

Widths, seed, common training parameters.

**Procedure**

1. Create dictionaries for models, histories, and evaluation results.
2. For each hidden width reset the PyTorch seed.
3. Instantiate the model and Adam optimizer, train for all epochs, then evaluate on the test loader.
4. Store model, loss history, and test result under the width key.

**Produces**

models, training_histories, model_evaluation_results.

**Acceptance contract**

Every width has 10 finite epoch losses and one held-out evaluation.

**Failure mode**

Report an interrupted run; do not compare an incomplete history to a full one.

## 12. Compare Learning Behaviour

**Purpose**

Compare measured optimization behaviour.

**Inputs**

Three histories and test evaluations.

**Procedure**

1. Plot training loss versus epoch for all widths.
2. Print each final training loss and test accuracy.
3. Keep reference reported losses separate from new measurements.
4. Follow the notebook's declared selection rule rather than changing it after seeing test results.

**Produces**

Loss curves and reported test accuracies.

**Acceptance contract**

Keep reference values separate from this run; selection uses lowest final training loss.

**Failure mode**

Do not select by test accuracy or present historical values as new measurements.

## 13. Visualize Predictions

**Purpose**

Inspect predictions and their confidence distribution.

**Inputs**

Each trained model and the first 15 test examples.

**Procedure**

1. Choose a fixed subset of test images.
2. For each model run inference and display predicted class/confidence against the true label.
3. Keep `model.eval()` active.
4. Save the prediction diagnostics.

**Produces**

Prediction grids and class-probability bars.

**Acceptance contract**

Same examples across widths; probabilities sum approximately to 1 per sample.

**Failure mode**

Investigate confidence/class mismatches before plotting conclusions.

## 14. Compute Confidence-Threshold Curves

**Purpose**

Describe exactly what confidence filtering measures.

**Inputs**

Aligned predicted classes, confidence and labels.

**Procedure**

1. Read predicted classes, confidence and labels from the same stored evaluation result; compute the boolean `correct` array.
2. Generate 100 thresholds between 0 and 0.99 and use the strict rule `confidence > threshold`.
3. Count accepted-correct (`TP`), accepted-incorrect (`FP`) and rejected-correct (`FN`) examples.
4. Compute `precision=TP/(TP+FP)`, `recall=TP/(TP+FN)` and the existing `accepted_accuracy=TP/N`.
5. Return NaN precision if none are accepted and zero recall if no predictions are correct; reorder all arrays consistently by recall.
6. Coverage would be `(TP+FP)/N`, but it is not returned by this implementation. Precision here is accuracy among accepted predictions; `accepted_accuracy` uses all test samples as its denominator.

**Produces**

Thresholds, precision, recall, accepted_accuracy arrays.

**Acceptance contract**

accepted means confidence>threshold; precision=accepted_correct/accepted; recall=accepted_correct/all_correct; accepted_accuracy=accepted_correct/N.

**Failure mode**

Precision is NaN when none are accepted; retain this undefined case. Coverage is not returned by the current helper.

## 15. Analyze the Selected Model

**Purpose**

Apply the declared selection rule to the confidence illustration.

**Inputs**

Final training losses and stored evaluations.

**Procedure**

1. Select `best_hidden_size` by the smallest final value in `training_histories`, as implemented.
2. Use that width's stored test result to compute confidence curves.
3. Save the precision–recall and accepted-correct/all-test-samples–recall plots.
4. Explain that these are correctness-retention curves, not per-digit precision–recall curves or evidence that confidence is calibrated.

**Produces**

best_hidden_size and confidence curves.

**Acceptance contract**

argmin final training loss determines width; all curve arrays are reordered together by recall.

**Failure mode**

Do not call this standard per-class precision/recall or claim the selected width is independently validated as optimal.

## 16. Run Final Validation

**Purpose**

Verify the completed experiment and saved figures.

**Inputs**

Datasets, three model results, histories and required files.

**Procedure**

1. Recheck dataset sizes and confirm all three widths were trained.
2. Verify every loss history has expected length and finite values.
3. Verify predictions/confidences/labels have matching lengths and valid ranges.
4. Verify every declared output exists before printing the final pass message.

**Produces**

Validation outcome and six figures.

**Acceptance contract**

Expected sample counts; complete finite histories; matching prediction arrays; valid confidence ranges.

**Failure mode**

Fail on missing models, invalid arrays or missing plots; undefined threshold precision is distinct from invalid model confidence.